# Functional tests

Below is a list of functional tests which should be included in sending and receiving implementations.

### Sending

- Ensure taproot outputs are excluded during coin selection if the sender does not have access to the key path private key (unless using H as the taproot internal key)
- Ensure the silent payment address is re-derived if inputs are added or removed during RBF

In [ ]:
# sending tests

import json

test_file = './test_vectors.json'

try: 
    with open(test_file, 'r') as f:
        print('Reading file...')
        data = json.load(f)
        print('File .json correctly read.')
except FileNotFoundError:
    print(f'Error: file {test_file} not found.')
except json.JSONDecodeError:
    print("Error: json file not valid.")

data = data[0]
sending = data['sending'][0]
sending_details = sending['given']
expected_outputs = sending['expected']['outputs']

vin = sending_details['vin']
recipients = sending_details['recipients'][0]

# check outputs
print(f'EXP Outputs: {expected_outputs}')
# display(vin)

from sender import *
# outputs = sending_run(vin, recipients)



inputs = select_inputs(vin)
# display(inputs)


# collect keys for valid inputs
keys = []
for tx in inputs:
    keys.append(int_from_hex(tx['private_key']))


# For each private key a_i corresponding to a BIP341 taproot output
# check that the private key produces a point with an even Y coordinate and negate the private key if not
for a_i in keys:
    P = pubkey_point_gen_from_int(a_i)
    if not has_even_y(P):
        keys.remove(a_i)

# let a = sum(a_i) ---> if a=0 fail
a = sum(keys)
if a == 0:
   raise ValueError('ERROR: zero key sum.')


# let input_hash = hashBIP0352/Inputs(outpointL || A)
# where outpointL is the smallest outpoint lexicographically used in the transaction and A = a·G
outpointL = get_outpoint()
A = point_mul(G, a) 
input_hash = tagged_hash('BIP0352/Inputs', outpointL + bytes_from_point(A)) 

# Group receiver silent payment addresses by B_scan (e.g. each group consists of one B_scan and one or more B_m)


In [ ]:
# sending tests

import json

test_file = './test_vectors.json'

try: 
    with open(test_file, 'r') as f:
        print('Reading file...')
        data = json.load(f)
        print('File .json correctly read.')
except FileNotFoundError:
    print(f'Error: file {test_file} not found.')
except json.JSONDecodeError:
    print("Error: json file not valid.")

data = data[0]
sending = data['sending'][0]
sending_details = sending['given']
expected_outputs = sending['expected']['outputs']

vin = sending_details['vin']
recipients = sending_details['recipients']

# check outputs
print(f'EXP Outputs: {expected_outputs}')
# display(vin)

from sender import *
# outputs = sending_run(vin, recipients)



inputs = select_inputs(vin)
# display(inputs)

# collect keys for valid inputs
keys = []
for tx in inputs:
    keys.append(int_from_hex(tx['private_key']))

# For each private key a_i corresponding to a BIP341 taproot output
# check that the private key produces a point with an even Y coordinate and negate the private key if not
for a_i in keys:
    P = pubkey_point_gen_from_int(a_i)
    if not has_even_y(P):
        keys.remove(a_i)

# let a = sum(a_i) ---> if a=0 fail
a = sum(keys)
if a == 0:
    raise ValueError('ERROR: zero key sum.')
A = point_mul(G, a) 

# let input_hash = hashBIP0352/Inputs(outpointL || A)
# where outpointL is the smallest outpoint lexicographically used in the transaction and A = a·G
outpointL = get_outpointL(inputs)
input_hash = tagged_hash('BIP0352/Inputs', outpointL + bytes_from_point(A)) 

recipients

outputs = []
# For each group:
for rp in recipients:
    # decode sp address
    _, data = decode(hrp='sp', addr=rp) 
    B_scan = data[:33]
    B_m = data[33:]

    

    # Let ecdh_shared_secret = input_hash·a·Bscan
    ecdh_shared_secret = point_mul(B_scan, int_from_bytes(input_hash) * a)
    # Let k = 0
    k = int(0)
    # Let tk = hashBIP0352/SharedSecret(serP(ecdh_shared_secret) || ser32(k))
    t_k = int_from_bytes(tagged_hash('BIP0352/SharedSecret', serP(ecdh_shared_secret) + ser32(k)))
    # If tk is not valid tweak, i.e., if tk = 0 or tk is larger or equal to the secp256k1 group order, fail
    if t_k == 0 or t_k >= n: 
        raise ValueError('ERROR') 
            
    # Let Pmn = Bm + tk·G
    P_mn = point_add(B_m, point_mul(G, t_k))
    # Encode Pmn as a BIP341 taproot output
    taproot_output = taproot_encode(P_mn)
    outputs.append(taproot_output)

    # Optionally, repeat with k++ to create additional outputs for the current Bm
    # If no additional outputs are required, continue to the next Bm with k++
    k += 1

outputs

### Receiving

- Ensure the public key can be extracted from non-standard P2PKH scriptSigs
- Ensure taproot script path spends are included, using the taproot output key (unless H is used as the taproot internal key)
- Ensure the scanner can extract the public key from each of the input types supported (e.g. P2WPKH, P2SH-P2WPKH, etc.)

In [3]:
# receiving tests

import json

test_file = './test_vectors.json'

try: 
    with open(test_file, 'r') as f:
        print('Reading file...')
        data = json.load(f)
        print('File .json correctly read.')
except FileNotFoundError:
    print(f'Error: file {test_file} not found.')
except json.JSONDecodeError:
    print("Error: json file not valid.")

data = data[0]
receiving = data['receiving'][0]
receiving_details = receiving['given']
expected_receiving = receiving['expected']

vin = receiving_details['vin']
outputs = receiving_details['outputs'][0]
key_material = receiving_details['key_material']
labels = receiving_details['labels']

expected_addresses = expected_receiving['addresses']
expected_outputs = expected_receiving['outputs'][0]


from receiver import receiving_run

address = receiving_run(key_material)

# check address
print(f'SIM Address: {address}')
print(f'EXP Address: {expected_addresses}')
print('ADDRESS CORRECT' if address == expected_addresses else 'WRONG ADDRESS')

# check outputs
print(f'EXP Outputs: {expected_outputs}')



Reading file...
File .json correctly read.
receiver.py loading...
SIM Address: ['sp1qqgste7k9hx0qftg6qmwlkqtwuy6cycyavzmzj85c6qdfhjdpdjtdgqjuexzk6murw56suy3e0rd2cgqvycxttddwsvgxe2usfpxumr70xc9pkqwv']
EXP Address: ['sp1qqgste7k9hx0qftg6qmwlkqtwuy6cycyavzmzj85c6qdfhjdpdjtdgqjuexzk6murw56suy3e0rd2cgqvycxttddwsvgxe2usfpxumr70xc9pkqwv']
ADDRESS CORRECT
EXP Outputs: {'priv_key_tweak': 'f438b40179a3c4262de12986c0e6cce0634007cdc79c1dcd3e20b9ebc2e7eef6', 'pub_key': '3e9fce73d4e77a4809908e3c3a2e54ee147b9312dc5044a193d1fc85de46e3c1', 'signature': '74f85b856337fbe837643b86f462118159f93ac4acc2671522f27e8f67b079959195ccc7a5dbee396d2909f5d680d6e30cda7359aa2755822509b70d6b0687a1'}
